![Noteable.ac.uk Banner](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/Images/Noteable%20NB%20Header%20Banner.png)

# From Classical NLP to Neural NLP: An Interactive Text Intelligence Notebook

## Legend

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>blue</b>, the <b>instructions</b> and <b>goals</b> are highlighted. This tells you what we are trying to achieve.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>green</b>, key <b>information</b> and <b>concept explanations</b> are highlighted.
</div>
<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>yellow</b>, <b>exercises</b> and <b>tasks</b> are highlighted for you to try yourself.
</div>
<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>red</b>, <b>error interpretation</b> and <b>debugging tips</b> are highlighted.
</div>

Welcome! In this notebook, we will build a **real text classification workflow** that moves from a **classical NLP baseline** to a **simple neural NLP model**.

Our task is **sentiment classification** using film reviews: can we predict whether a review is **positive** or **negative**?

This is a great teaching example because it shows something important about modern AI work:
- strong **classical baselines** still matter
- careful **evaluation** still matters
- and even in the age of larger neural models, understanding the fundamentals is a superpower

By the end, you will have:
- explored a real text dataset
- cleaned text with **spaCy**
- trained a **TF-IDF + Logistic Regression** model with **scikit-learn**
- trained a lightweight **PyTorch** neural classifier
- compared both approaches using proper metrics
- tested both models in an interactive prediction playground


## 1. Setting Up Our Toolkit

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Import the libraries we need for text processing, machine learning, visualisation, and interactivity. We will also check that our text dataset is available.
</div>

Click the code cell below and press the **<code>▶</code> Run** button in the toolbar (or use <code>Shift + Enter</code>). The print messages will help you track our progress.

In [ ]:
print("Starting setup: importing libraries and checking the environment...")

import warnings
import random
import copy
from collections import Counter

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import movie_reviews

import spacy

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

import plotly.express as px
import plotly.graph_objects as go

import ipywidgets as widgets
from IPython.display import display, clear_output

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    movie_reviews.categories()
    print("NLTK movie_reviews corpus is available.")
except LookupError:
    print("movie_reviews corpus not found locally. Downloading now...")
    nltk.download("movie_reviews")
    print("movie_reviews corpus downloaded.")

print(f"PyTorch device in use: {device}")
print("Success! Imports and environment checks are complete.")

## 2. Loading a Lightweight Text Dataset

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Load a reliable text classification dataset that works well in teaching notebooks. We will use the built-in <b>NLTK movie_reviews</b> corpus, which contains positive and negative film reviews.
</div>

This keeps the notebook practical and robust: no credentials, no fragile scraping, and no large external download.

In [ ]:
print("Loading movie review documents...")

records = []
for label in movie_reviews.categories():
    for fileid in movie_reviews.fileids(label):
        records.append(
            {
                "fileid": fileid,
                "label": label,
                "label_id": 1 if label == "pos" else 0,
                "text": movie_reviews.raw(fileid),
            }
        )

df = pd.DataFrame(records).sample(frac=1, random_state=SEED).reset_index(drop=True)

# EDIT THIS VALUE: set to a smaller number like 600 if you want a faster experiment.
SAMPLE_SIZE = None
if SAMPLE_SIZE is not None:
    df = df.head(SAMPLE_SIZE).copy()

df["char_count"] = df["text"].str.len()
df["word_count_raw"] = df["text"].str.split().str.len()

print(f"Dataset loaded successfully with {len(df)} reviews.")
print("Unique class labels:", sorted(df["label"].unique().tolist()))
display(df[["label", "fileid", "text"]].head(3))

## 3. First Look at the Data

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Why this matters:</b> Before we model anything, we should understand the dataset. Are the classes balanced? Are some reviews much longer than others? These simple checks help us avoid poor assumptions later.
</div>

Run the next cell to inspect class balance and review lengths.

In [ ]:
print("Exploring class balance and review length...")

class_counts = df["label"].value_counts().rename_axis("label").reset_index(name="count")

fig_balance = px.bar(
    class_counts,
    x="label",
    y="count",
    color="label",
    text="count",
    title="Class balance in the movie review dataset",
    color_discrete_map={"neg": "#C44E52", "pos": "#55A868"},
)
fig_balance.update_traces(textposition="outside")
fig_balance.update_layout(showlegend=False, xaxis_title="Sentiment", yaxis_title="Number of reviews")
fig_balance.show()

fig_length = px.histogram(
    df,
    x="word_count_raw",
    color="label",
    nbins=40,
    opacity=0.75,
    marginal="box",
    title="Review length distribution (raw word counts)",
    color_discrete_map={"neg": "#C44E52", "pos": "#55A868"},
)
fig_length.update_layout(xaxis_title="Words per review", yaxis_title="Number of reviews")
fig_length.show()

summary = df.groupby("label")[["word_count_raw", "char_count"]].agg(["mean", "median"]).round(1)
display(summary)

## 4. Text Preprocessing with spaCy

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>spaCy preprocessing:</b> Raw text is messy. We will tokenize each review, remove punctuation and stop words, keep alphabetic terms, and use lemmatisation where available. This helps both the classical and neural workflows start from cleaner text.
</div>

Even though modern models can sometimes work directly with raw text, it is still valuable to understand how text cleaning changes what the model sees.

In [ ]:
print("Loading spaCy and preprocessing text...")

try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "textcat"])
    print("Loaded spaCy model: en_core_web_sm")
except OSError:
    print("spaCy model not found, so a lightweight blank English pipeline will be used instead.")
    nlp = spacy.blank("en")

def doc_to_tokens(doc):
    cleaned_tokens = []
    for token in doc:
        if token.is_space or token.is_punct or token.like_num:
            continue
        if not token.is_alpha or token.is_stop:
            continue
        lemma = token.lemma_.lower().strip() if token.lemma_ else token.lower_.strip()
        if not lemma or lemma == "-pron-":
            lemma = token.lower_.strip()
        if len(lemma) < 3:
            continue
        cleaned_tokens.append(lemma)
    return cleaned_tokens or ["empty"]

processed_tokens = [doc_to_tokens(doc) for doc in nlp.pipe(df["text"], batch_size=64)]
df["tokens"] = processed_tokens
df["processed_text"] = [" ".join(tokens) for tokens in processed_tokens]
df["token_count"] = df["tokens"].str.len()

print("Preprocessing complete.")
display(df[["label", "tokens", "processed_text"]].head(3))

In [ ]:
# WILL TAKE A WHILE (~20 seconds)
print("Visualising common cleaned tokens...")

token_counts = Counter(token for tokens in df["tokens"] for token in tokens)
common_tokens_df = pd.DataFrame(token_counts.most_common(15), columns=["token", "count"])

fig_tokens = px.bar(
    common_tokens_df,
    x="count",
    y="token",
    orientation="h",
    color="count",
    color_continuous_scale="Blues",
    title="Most common tokens after spaCy preprocessing",
)
fig_tokens.update_layout(
    yaxis=dict(categoryorder="total ascending"),
    xaxis_title="Count",
    yaxis_title="Token",
    coloraxis_showscale=False,
)
fig_tokens.show()

## 5. A Classical NLP Baseline with scikit-learn

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Build a strong classical baseline using <b>TF-IDF</b> features and <b>Logistic Regression</b>. This is a very common and surprisingly effective approach for text classification.
</div>

We will use a **train / validation / test** split. That means:
- the model learns from the training set
- we monitor decisions on the validation set
- and we keep the test set untouched until final evaluation

This is good machine learning practice.

In [ ]:
print("Creating train, validation, and test splits...")

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label_id"],
    random_state=SEED,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label_id"],
    random_state=SEED,
)

print(f"Train size: {len(train_df)} | Validation size: {len(val_df)} | Test size: {len(test_df)}")

def evaluate_model(y_true, y_pred, split_name):
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")
    return {
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# EDIT THIS VALUE: increase or decrease the size of the TF-IDF vocabulary.
MAX_FEATURES = 8000

vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1, 2), min_df=2)
X_train = vectorizer.fit_transform(train_df["processed_text"])
X_val = vectorizer.transform(val_df["processed_text"])
X_test = vectorizer.transform(test_df["processed_text"])

classical_model = LogisticRegression(max_iter=1000, random_state=SEED)
classical_model.fit(X_train, train_df["label_id"])

classical_val_pred = classical_model.predict(X_val)
classical_test_pred = classical_model.predict(X_test)
classical_val_proba = classical_model.predict_proba(X_val)
classical_test_proba = classical_model.predict_proba(X_test)

classical_val_metrics = evaluate_model(val_df["label_id"], classical_val_pred, "validation")
classical_test_metrics = evaluate_model(test_df["label_id"], classical_test_pred, "test")
classical_cm = confusion_matrix(test_df["label_id"], classical_test_pred)

print("Classical baseline trained and evaluated.")
print("Validation metrics:", {k: round(v, 3) if isinstance(v, float) else v for k, v in classical_val_metrics.items()})
print("Test classification report:")
print(classification_report(test_df["label_id"], classical_test_pred, target_names=["negative", "positive"]))

In [ ]:
print("Visualising classical model results...")

cm_df = pd.DataFrame(
    classical_cm,
    index=["True negative", "True positive"],
    columns=["Pred negative", "Pred positive"],
)

fig_cm = px.imshow(
    cm_df,
    text_auto=True,
    color_continuous_scale="Blues",
    aspect="auto",
    title="Classical model confusion matrix (test set)",
)
fig_cm.update_layout(coloraxis_showscale=False)
fig_cm.show()

feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = classical_model.coef_[0]

top_positive = pd.DataFrame(
    {
        "term": feature_names[np.argsort(coefficients)[-10:]],
        "weight": np.sort(coefficients)[-10:],
        "direction": "Positive signal",
    }
)
top_negative = pd.DataFrame(
    {
        "term": feature_names[np.argsort(coefficients)[:10]],
        "weight": np.sort(coefficients)[:10],
        "direction": "Negative signal",
    }
)
top_features_df = pd.concat([top_negative, top_positive], ignore_index=True)

fig_features = px.bar(
    top_features_df.sort_values("weight"),
    x="weight",
    y="term",
    color="direction",
    orientation="h",
    title="Words and phrases the logistic regression model relies on",
    color_discrete_map={"Negative signal": "#C44E52", "Positive signal": "#55A868"},
)
fig_features.update_layout(xaxis_title="Model weight", yaxis_title="Token / n-gram")
fig_features.show()

## 6. A Simple Neural NLP Model with PyTorch

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Train a lightweight neural text classifier in <b>PyTorch</b>. We will build a simple embedding-based model that learns a vector representation for words, averages them across each review, and then predicts sentiment.
</div>

This is not a giant language model, and that is intentional. The aim here is to show the jump from a classical pipeline to a neural one in a way that is still fast, clear, and runnable on CPU.

In [ ]:
print("Preparing neural text data and building a PyTorch classifier...")

# EDIT THIS VALUE: maximum vocabulary size for the neural model.
MAX_VOCAB = 8000
# EDIT THIS VALUE: maximum number of tokens kept per review.
MAX_LEN = 120
# EDIT THIS VALUE: embedding dimension.
EMBED_DIM = 64
# EDIT THIS VALUE: hidden layer width.
HIDDEN_DIM = 64
# EDIT THIS VALUE: batch size.
BATCH_SIZE = 64
# EDIT THIS VALUE: number of epochs.
EPOCHS = 6
LEARNING_RATE = 0.001

counter = Counter(token for tokens in train_df["tokens"] for token in tokens)
most_common = counter.most_common(MAX_VOCAB - 2)
vocab = {"<PAD>": 0, "<UNK>": 1}
for token, _ in most_common:
    vocab[token] = len(vocab)

def encode_tokens(tokens, vocab_map, max_len):
    ids = [vocab_map.get(token, vocab_map["<UNK>"]) for token in tokens[:max_len]]
    if len(ids) == 0:
        ids = [vocab_map["<UNK>"]]
    if len(ids) < max_len:
        ids += [vocab_map["<PAD>"]] * (max_len - len(ids))
    return ids

class ReviewDataset(Dataset):
    def __init__(self, frame, vocab_map, max_len):
        self.sequences = [encode_tokens(tokens, vocab_map, max_len) for tokens in frame["tokens"]]
        self.labels = frame["label_id"].tolist()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return (
            torch.tensor(self.sequences[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )

train_dataset = ReviewDataset(train_df, vocab, MAX_LEN)
val_dataset = ReviewDataset(val_df, vocab, MAX_LEN)
test_dataset = ReviewDataset(test_df, vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

class AverageEmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=2, padding_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.hidden = nn.Linear(embed_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.2)
        self.activation = nn.ReLU()
    def forward(self, x):
        embedded = self.embedding(x)
        mask = (x != 0).unsqueeze(-1)
        pooled = (embedded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        hidden = self.dropout(self.activation(self.hidden(pooled)))
        return self.output(hidden)

neural_model = AverageEmbeddingClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(neural_model.parameters(), lr=LEARNING_RATE)

def evaluate_loader(model, loader):
    model.eval()
    losses = []
    all_labels = []
    all_preds = []
    all_probs = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            losses.append(loss.item())
            all_labels.extend(yb.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
    metrics = evaluate_model(all_labels, all_preds, "loader")
    return np.mean(losses), metrics, np.array(all_probs), np.array(all_preds), np.array(all_labels)

history = []
best_state = copy.deepcopy(neural_model.state_dict())
best_val_f1 = -1

print("Training neural model...")
for epoch in range(1, EPOCHS + 1):
    neural_model.train()
    batch_losses = []
    all_train_labels = []
    all_train_preds = []
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = neural_model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
        preds = logits.argmax(dim=1)
        all_train_labels.extend(yb.detach().cpu().numpy().tolist())
        all_train_preds.extend(preds.detach().cpu().numpy().tolist())

    train_acc = accuracy_score(all_train_labels, all_train_preds)
    val_loss, val_metrics, _, _, _ = evaluate_loader(neural_model, val_loader)

    history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(batch_losses)),
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_f1": val_metrics["f1"],
        }
    )

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_state = copy.deepcopy(neural_model.state_dict())

    print(
        f"Epoch {epoch}/{EPOCHS} | train loss: {np.mean(batch_losses):.4f} | train acc: {train_acc:.3f} | val acc: {val_metrics['accuracy']:.3f} | val f1: {val_metrics['f1']:.3f}"
    )

neural_model.load_state_dict(best_state)
test_loss, neural_test_metrics, neural_test_proba, neural_test_pred, neural_test_true = evaluate_loader(neural_model, test_loader)
neural_test_metrics["split"] = "test"
neural_cm = confusion_matrix(neural_test_true, neural_test_pred)

print("Neural model training complete.")
print("Best validation F1:", round(best_val_f1, 3))
print("Neural test metrics:", {k: round(v, 3) if isinstance(v, float) else v for k, v in neural_test_metrics.items()})
print("Test classification report:")
print(classification_report(neural_test_true, neural_test_pred, target_names=["negative", "positive"]))

In [ ]:
print("Visualising neural training history and test results...")

history_df = pd.DataFrame(history)

fig_history = go.Figure()
fig_history.add_trace(go.Scatter(x=history_df["epoch"], y=history_df["train_accuracy"], mode="lines+markers", name="Train accuracy"))
fig_history.add_trace(go.Scatter(x=history_df["epoch"], y=history_df["val_accuracy"], mode="lines+markers", name="Validation accuracy"))
fig_history.add_trace(go.Scatter(x=history_df["epoch"], y=history_df["val_f1"], mode="lines+markers", name="Validation F1"))
fig_history.update_layout(
    title="Neural model learning curve",
    xaxis_title="Epoch",
    yaxis_title="Score",
    yaxis=dict(range=[0, 1]),
)
fig_history.show()

neural_cm_df = pd.DataFrame(
    neural_cm,
    index=["True negative", "True positive"],
    columns=["Pred negative", "Pred positive"],
)

fig_neural_cm = px.imshow(
    neural_cm_df,
    text_auto=True,
    color_continuous_scale="Greens",
    aspect="auto",
    title="Neural model confusion matrix (test set)",
)
fig_neural_cm.update_layout(coloraxis_showscale=False)
fig_neural_cm.show()

## 7. Direct Comparison: Classical vs Neural NLP

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Key idea:</b> Neural models are powerful, but they do not automatically beat classical models on every task. A strong baseline gives us something meaningful to compare against.
</div>

In many real projects, this exact comparison is essential. If a simpler model performs just as well, it may be easier to train, interpret, and deploy.

In [ ]:
print("Comparing the classical and neural approaches...")

comparison_df = pd.DataFrame(
    [
        {
            "model": "Classical TF-IDF + Logistic Regression",
            "accuracy": classical_test_metrics["accuracy"],
            "precision": classical_test_metrics["precision"],
            "recall": classical_test_metrics["recall"],
            "f1": classical_test_metrics["f1"],
        },
        {
            "model": "Neural Embedding Averager (PyTorch)",
            "accuracy": neural_test_metrics["accuracy"],
            "precision": neural_test_metrics["precision"],
            "recall": neural_test_metrics["recall"],
            "f1": neural_test_metrics["f1"],
        },
    ]
).round(3)

display(comparison_df)

comparison_long = comparison_df.melt(id_vars="model", var_name="metric", value_name="score")
fig_compare = px.bar(
    comparison_long,
    x="metric",
    y="score",
    color="model",
    barmode="group",
    text="score",
    title="Test-set comparison: classical vs neural NLP",
)
fig_compare.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig_compare.update_layout(yaxis=dict(range=[0, 1]))
fig_compare.show()

results_df = test_df[["text", "label", "label_id"]].copy().reset_index(drop=True)
results_df["classical_pred_id"] = classical_test_pred
results_df["neural_pred_id"] = neural_test_pred
results_df["classical_confidence"] = classical_test_proba.max(axis=1)
results_df["neural_confidence"] = neural_test_proba.max(axis=1)

id_to_label = {0: "negative", 1: "positive"}
results_df["classical_pred"] = results_df["classical_pred_id"].map(id_to_label)
results_df["neural_pred"] = results_df["neural_pred_id"].map(id_to_label)
results_df["models_disagree"] = results_df["classical_pred_id"] != results_df["neural_pred_id"]
results_df["either_wrong"] = (
    (results_df["classical_pred_id"] != results_df["label_id"]) |
    (results_df["neural_pred_id"] != results_df["label_id"])
)

disagreement_view = results_df.loc[
    results_df["models_disagree"] | results_df["either_wrong"],
    ["label", "classical_pred", "neural_pred", "classical_confidence", "neural_confidence", "text"],
].head(8)

print("A few interesting test examples where at least one model struggled:")
display(disagreement_view)

## 8. Interactive Prediction Playground

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Try it yourself:</b> Use the widget below to test both models on your own review text. This is a great way to see where the models agree, where they differ, and how confident they seem.
</div>

You can choose a preset example or type your own short film review. Then press the button to generate predictions.

In [ ]:
print("Building the interactive prediction playground...")

label_names = {0: "negative", 1: "positive"}

def preprocess_text_for_models(text):
    doc = next(nlp.pipe([text], batch_size=1))
    tokens = doc_to_tokens(doc)
    if not tokens:
        tokens = ["<UNK>"]
    return tokens, " ".join(tokens)

def predict_with_both_models(text):
    tokens, processed = preprocess_text_for_models(text)
    classical_probs = classical_model.predict_proba(vectorizer.transform([processed]))[0]

    encoded = torch.tensor([encode_tokens(tokens, vocab, MAX_LEN)], dtype=torch.long).to(device)
    neural_model.eval()
    with torch.no_grad():
        neural_probs = torch.softmax(neural_model(encoded), dim=1)[0].cpu().numpy()

    summary_df = pd.DataFrame(
        {
            "model": ["Classical logistic regression", "Neural PyTorch classifier"],
            "predicted_label": [label_names[int(np.argmax(classical_probs))], label_names[int(np.argmax(neural_probs))]],
            "confidence": [float(np.max(classical_probs)), float(np.max(neural_probs))],
        }
    )

    prob_df = pd.DataFrame(
        {
            "sentiment": ["negative", "positive", "negative", "positive"],
            "probability": [classical_probs[0], classical_probs[1], neural_probs[0], neural_probs[1]],
            "model": ["Classical", "Classical", "Neural", "Neural"],
        }
    )
    return tokens, summary_df, prob_df

preset_examples = [
    ("Choose a preset example...", ""),
    ("Positive review example", "A funny, warm film with great performances and a satisfying ending."),
    ("Negative review example", "This movie felt slow, messy, and forgettable despite its talented cast."),
    ("Mixed review example", "The acting is strong, but the story drifts and the ending is disappointing."),
]

preset_dropdown = widgets.Dropdown(options=preset_examples, description="Preset:")
text_box = widgets.Textarea(
    value="A smart, tense thriller with excellent dialogue and memorable characters.",
    description="Review:",
    layout=widgets.Layout(width="100%", height="140px"),
    style={"description_width": "60px"},
)
predict_button = widgets.Button(description="Predict sentiment", button_style="success", icon="search")
playground_output = widgets.Output()

def load_preset(change):
    if change["name"] == "value" and change["new"]:
        text_box.value = change["new"]

preset_dropdown.observe(load_preset, names="value")

def run_prediction(_):
    with playground_output:
        clear_output(wait=True)
        user_text = text_box.value.strip()
        if not user_text:
            print("Please enter some review text first.")
            return

        tokens, summary_df, prob_df = predict_with_both_models(user_text)
        print("Cleaned tokens used by the models:")
        print(tokens[:25], "..." if len(tokens) > 25 else "")
        display(summary_df.round(3))

        fig = px.bar(
            prob_df,
            x="sentiment",
            y="probability",
            color="model",
            barmode="group",
            text="probability",
            title="How confident is each model?",
        )
        fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
        fig.update_layout(yaxis=dict(range=[0, 1]))
        fig.show()

predict_button.on_click(run_prediction)

display(widgets.VBox([preset_dropdown, text_box, predict_button, playground_output]))
print("Interactive demo ready! Try a preset review or type your own text, then press the button.")

## 9. Limitations, Failure Cases, and Responsible Use

<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Be careful:</b> Good notebook results do not mean perfect understanding. Text models can be overconfident, brittle, and sensitive to wording. Responsible use means checking where they fail and thinking carefully about context.
</div>

A few important limits to keep in mind:

- **Small benchmark task:** This dataset is useful for teaching, but it is much simpler than many real-world NLP problems.
- **Domain dependence:** A model trained on film reviews may behave poorly on product reviews, social media posts, or academic writing.
- **Subtle language:** Sarcasm, negation, mixed sentiment, and cultural references can still confuse both models.
- **Confidence is not certainty:** A high predicted probability does not guarantee the model is right.
- **Bias and representation:** Training data reflects human choices and historical context. Models can learn unwanted patterns from that data.

In practice, responsible NLP work usually includes:
- testing on data from the real deployment setting
- checking errors by hand
- comparing multiple models, not just one
- documenting limitations clearly for users and stakeholders


## Take-away

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    Congratulations! You have built an end-to-end text intelligence workflow: you loaded data, explored its structure, cleaned text with spaCy, trained a strong classical baseline, trained a neural PyTorch model, compared their results, and tested both interactively. This is exactly the kind of workflow that supports thoughtful, evidence-based NLP practice.
</div>

### Suggested next steps
- Try changing the editable values such as <code>MAX_FEATURES</code>, <code>MAX_VOCAB</code>, or <code>EPOCHS</code>
- Inspect more failures in the test set
- Replace the neural architecture with an LSTM or a 1D CNN
- Compare lemmatised text with raw text
- Extend the notebook to a multi-class topic classification task


![Noteable license](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/Images/Noteable%20Notebook%20Footer.png)